In [1]:
import Pkg
Pkg.update()
Pkg.add("ProfileView")
Pkg.add("ProfileCanvas")

Pkg.Types.PkgError: The active project has been set to a file that isn't a Project file: C:\Users\micha\.julia\juliaup\julia-1.12.5+0.x64.w64.mingw32\bin\julia.exe
The project path must be to a Project file or directory.


In [1]:
import Pkg
using MLDatasets
using Flux: onehotbatch, onecold
using Statistics
using LinearAlgebra
using Random
using ProfileCanvas
using InteractiveUtils

train_data = MLDatasets.FashionMNIST(split=:train)
test_data  = MLDatasets.FashionMNIST(split=:test)

X_train = Float32.(reshape(train_data.features, 28, 28, 1, :))
X_test  = Float32.(reshape(test_data.features, 28, 28, 1, :))

Y_train_raw = train_data.targets
Y_test_raw  = test_data.targets

Y_train = Float32.(onehotbatch(Y_train_raw, 0:9))
Y_test  = Float32.(onehotbatch(Y_test_raw, 0:9))

println("Wymiary: ", size(X_train))

include("clothesolver.jl") 

# 1. Definicja
my_net_def = Chain(
  Conv((3, 3), 1 => 6, bias=false),
  MaxPool((2, 2)),
  Conv((3, 3), 6 => 16, bias=false),
  MaxPool((2, 2)),
  Flatten(),               
  Dense(400 => 84, relu), # Pamiętaj o 400 z Shape Inference!
  Dropout(0.4),
  Dense(84 => 10)
)

# 2. Alokacja Memory Pool
my_model = build_model(my_net_def, (28, 28, 1))

input_node  = alloc_act!(my_model.pool, 28, 28, 1)
target_node = alloc_act!(my_model.pool, 10)
loss_fn     = LogitCrossEntropy(my_model.pool, 10)

Wymiary: (28, 28, 1, 60000)


LogitCrossEntropy{Base.ReshapedArray{Float32, 2, SubArray{Float32, 1, Vector{Float32}, Tuple{UnitRange{Int64}}, true}, Tuple{}}, SubArray{Float32, 1, Vector{Float32}, Tuple{UnitRange{Int64}}, true}}(GraphNode{Base.ReshapedArray{Float32, 2, SubArray{Float32, 1, Vector{Float32}, Tuple{UnitRange{Int64}}, true}, Tuple{}}}(Float32[0.0; 0.0; … ; 0.0; 0.0;;], Float32[0.0; 0.0; … ; 0.0; 0.0;;]), GraphNode{SubArray{Float32, 1, Vector{Float32}, Tuple{UnitRange{Int64}}, true}}(Float32[0.0], Float32[0.0]))

In [5]:
bs = my_model.batch_size

x_flat = parent(input_node.data)
y_flat = parent(target_node.data)

len_x_batch = 28 * 28 * 1 * bs
len_y_batch = 10 * bs

@inbounds @simd for k in 1:len_x_batch
    x_flat[k] = X_train[k]
end
@inbounds @simd for k in 1:len_y_batch
    y_flat[k] = Y_train[k]
end

preds = forward!(my_model.chain, input_node, true)
primal!(loss_fn, preds, target_node)
loss_fn.out.grad[1] = 1.0f0
adjoint!(loss_fn, preds, target_node)
backward!(my_model.chain, input_node, true)

println("Kompilacja zakończona, start profilowania...")

@time ProfileCanvas.@profview for _ in 1:1000
    zero_a_grad!(my_model.pool)
    zero_w_grad!(my_model.pool)
    
    @inbounds @simd for k in 1:len_x_batch
        x_flat[k] = X_train[k]
    end
    @inbounds @simd for k in 1:len_y_batch
        y_flat[k] = Y_train[k]
    end
    
    preds = forward!(my_model.chain, input_node, true)
    primal!(loss_fn, preds, target_node)
    
    loss_fn.out.grad[1] = 1.0f0
    adjoint!(loss_fn, preds, target_node)
    backward!(my_model.chain, input_node, true)
end

Kompilacja zakończona, start profilowania...
  0.332822 seconds (1.38 M allocations: 37.318 MiB, 2.28% gc time)


In [ ]:
println("============== ANALIZA primal! DLA WSZYSTKICH WARSTW ==============")

curr_x = my_model.input 

for (i, layer) in enumerate(my_model.chain.layers)
    layer_name = typeof(layer).name.name
    println("\n\n" * "="^10 * " Warstwa $i: $layer_name " * "="^10)
    code_warntype(primal!, (typeof(layer), typeof(curr_x), Bool))
    curr_x = layer.out 
end

============== ANALIZA primal! DLA WSZYSTKICH WARSTW ==============


========== Warstwa 1: ConvLayer ==========
MethodInstance for primal!(::ConvLayer{Base.ReshapedArray{Float32, 4, SubArray{Float32, 1, Vector{Float32}, Tuple{UnitRange{Int64}}, true}, Tuple{}}, SubArray{Float32, 1, Vector{Float32}, Tuple{UnitRange{Int64}}, true}, Base.ReshapedArray{Float32, 4, SubArray{Float32, 1, Vector{Float32}, Tuple{UnitRange{Int64}}, true}, Tuple{}}, Matrix{Float32}}, ::GraphNode{Base.ReshapedArray{Float32, 4, SubArray{Float32, 1, Vector{Float32}, Tuple{UnitRange{Int64}}, true}, Tuple{}}}, ::Bool)
  from primal!(layer::ConvLayer, x::GraphNode, is_training::Bool) @ Main c:\Users\micha\Desktop\CNNetwork\clothesolver.jl:142
Arguments
  #self#::Core.Const(Main.primal!)
  layer::ConvLayer{Base.ReshapedArray{Float32, 4, SubArray{Float32, 1, Vector{Float32}, Tuple{UnitRange{Int64}}, true}, Tuple{}}, SubArray{Float32, 1, Vector{Float32}, Tuple{UnitRange{Int64}}, true}, Base.ReshapedArray{Float32, 4, SubA